# S0 — Instrument & Audio Diagnostics (single identity: `May`)**Question this notebook answers:** does the Run-E-style predictor use audio at all,measured on frames it has never seen?Everything here is pre-registered before any number is computed, and every headlinenumber is reported against three nulls (persistence, shuffled audio, zeroed audio).No architecture changes. If audio fails here, no decoder is worth building.---## Three things found in the repo that this notebook fixesThese were found by reading `talking_head_jepa.py` and`JEPA_TalkingHead_EncoderRetrain.ipynb` on 2026-09-03. They are load-bearing.**1. `Run D`/`Run E` had no train/test split at all.**`SyncTalkDatasetFixed.__len__` returns `len(image_paths) - seq_len`, and the`DataLoader` shuffles over *all* of it. The official SyncTalk split for `May`(`transforms_val.json`) is frames **5520–6072**, and those frames were in thetraining set. Every rollout number reported for Run D/E on `May` is therefore**in-sample**. This notebook retrains the predictor side on frames 0–5519 only(call it **Run F**) so that a held-out number exists.**2. `TalkingHeadJEPA.forward()` in `talking_head_jepa.py` mis-aligns audio.**It does `ctx_act = act_emb[:, :ctx_len]` and `tgt_emb = emb[:, num_preds:]` — thepredictor is asked for frames 4–7 while conditioned on the audio of frames 0–3.The `EncoderRetrain` notebook does *not* use that method; it uses`pred_act = act_emb[:, ctx_len:ctx_len+num_preds]` (correct, future audio). SoRun D/E are fine — but anyone who calls `model.forward()` silently gets thebroken alignment. Do not use that method. This notebook re-implements thecorrect loss inline.**3. The FaRL latent is 768-d, not 196x768.**`FarlFrozenFaceEncoder.forward()` returns `x[:, 1:].mean(dim=1)` — the *mean* overpatch tokens, placed at sequence position 0. Notes claiming "196 tokens x 768 dims= 150,528 floats = no compression bottleneck" describe a pipeline that does notexist. The real per-frame latent is **768 floats for a 150,528-value image: 196xcompression**. `S1_Decoder_Oracle.ipynb` measures both.## The change that makes this loop fastThe face encoder is *frozen*. Its output is therefore a pure function of the frame,so it can be computed once for all 6073 frames and cached. Everything that trains(`projector`, `audio_embedder`, `predictor`, `pred_proj`) then trains on 768-dvectors instead of images. A Run-E-equivalent 15-epoch run drops from ~4 GPU-hoursto a couple of minutes, which is what makes pre-registered controls affordable at all.

## 0. Setup

In [ ]:
from google.colab import drivedrive.mount('/content/drive')

In [ ]:
import os, sys, glob, json, time, gc, math, hashlib, tarfile, shutilimport numpy as npimport torchimport torch.nn as nnimport torch.nn.functional as Fimport cv2import matplotlib.pyplot as pltfrom einops import rearrangefrom tqdm.auto import tqdmCODE_BASE = '/content/drive/MyDrive/Data/le-wm'   # adjust if your repo lives elsewhereassert os.path.isdir(CODE_BASE), f'{CODE_BASE} not found - check the Drive path'os.chdir(CODE_BASE)sys.path.insert(0, CODE_BASE)from module import MLP, ARPredictor          # repo modules, unmodifiedfrom jepa import JEPA                        # imported for reference; not used directly belowdevice = torch.device('cuda' if torch.cuda.is_available() else 'cpu')print('device:', device)if device.type != 'cuda':    print('WARNING: no GPU. The FaRL caching pass will be slow (but only runs once).')print('torch', torch.__version__)

In [ ]:
# --- Config -------------------------------------------------------------------IDENTITY   = 'May'DATA_ROOT  = os.path.join(CODE_BASE, 'data')LOCAL_ROOT = '/content/local_data'LOCAL_PATH = os.path.join(LOCAL_ROOT, IDENTITY)CACHE_DIR  = os.path.join(CODE_BASE, 'study_cache');   os.makedirs(CACHE_DIR, exist_ok=True)PREREG_DIR = os.path.join(CODE_BASE, 'prereg');        os.makedirs(PREREG_DIR, exist_ok=True)RESULTS_DIR= os.path.join(CODE_BASE, 'study_results'); os.makedirs(RESULTS_DIR, exist_ok=True)CKPT_DIR   = os.path.join(CODE_BASE, 'checkpoints');   os.makedirs(CKPT_DIR, exist_ok=True)STUDY_ID   = 'S0_audio_instrument'RUN_TAG    = 'runF_trainblock_only'   # Run E config, but trained on frames 0..5519 only# model geometry - identical to Run D/E (config/train/talking_head.yaml + the 512 override)IMAGE_SIZE   = 224AUDIO_DIM    = 512     # aud_ave.npy is 512-d per frame; Run D/E used one frame (0.2 s), no windowEMBED_DIM    = 256HIDDEN_DIM   = 256HISTORY_SIZE = 4NUM_PREDS    = 4N_HOPS_TRAIN = 3       # Run E scheduled sampling: 3 chained hops = 12 future framesSEQ_LEN      = HISTORY_SIZE + N_HOPS_TRAIN * NUM_PREDS   # 16N_EPOCHS   = 15        # same as Run D/EBATCH_SIZE = 8LR         = 1e-4SEED       = 0torch.manual_seed(SEED); np.random.seed(SEED)print(f'{STUDY_ID} / {RUN_TAG}: seq_len={SEQ_LEN}, embed={EMBED_DIM}, audio={AUDIO_DIM}')

## 1. Pre-registrationThis cell is **write-once**. It records the endpoints, the nulls, and the decisionthresholds *before* any result exists. If the file is already there it is loaded andleft alone — so re-running the notebook cannot quietly move a goalpost. To change thepre-registration you must delete the file by hand, which leaves a visible trace.The thresholds below come from the study plan; the primary endpoint is the only onethat decides go/no-go.

In [ ]:
PREREG_PATH = os.path.join(PREREG_DIR, f'{STUDY_ID}.json')PREREG = {    'study_id': STUDY_ID,    'run_tag': RUN_TAG,    'question': 'Does the Run-E-architecture predictor use audio, on held-out frames?',    'split': {        'kind': 'contiguous, official SyncTalk split',        'source': 'data/May/transforms_train.json / transforms_val.json',        'train_frames': '0..5519', 'test_frames': '5520..6072',        'why': ('random frame splits leak: at 25 fps a random held-out frame sits ~40-280 ms '                'from a training frame and is effectively in-sample. Reproduced earlier: '                'test->nearest-train mean 2.52 frames, max 7, 100% within 280 ms.'),    },    'primary_endpoint': {        'name': 'shuffled_audio_delta_pct_hop1_heldout',        'definition': '100 * (MSE_shuffled_audio - MSE_real_audio) / MSE_real_audio, '                      'hop-1 (4-step) prediction, test frames only, real context',        'pass': '>= 20 with bootstrap 95% CI lower bound > 5',        'weak': '5 <= delta < 20, or CI lower bound <= 5',        'fail': '< 5',        'decision': 'FAIL => fix conditioning (whitening, audio windowing, residual target) '                    'before any decoder work. PASS/WEAK => S1 decoder oracle may proceed.',    },    'secondary_endpoints': [        'shuffled_audio_delta_pct_hop3_heldout',        'zeroed_audio_delta_pct_heldout',        'audio_share = (MSE_shuffled - MSE_real) / (MSE_persistence - MSE_real)',        'condition_driven_fraction on gate rows of every predictor block',        'in_sample_vs_heldout gap on the same metric',    ],    'nulls': {        'persistence': 'copy the last real context embedding for all predicted steps',        'train_mean': 'predict the training-block mean embedding',        'shuffled_audio': 'audio rows permuted across the WHOLE test block (destroys '                          'frame-audio correspondence, preserves marginal statistics)',        'zeroed_audio': 'audio replaced by zeros',    },    'thresholds': {        'cdf_dead': 0.02, 'cdf_weak': 0.15, 'cdf_alive': 0.30,        'persistence_share_floor': 0.05,        'delta_pass': 20.0, 'delta_weak': 5.0,    },    'known_traps': [        'A fixed random vector per frame index carries zero information yet reached +7% delta '        'and CDF 0.34 in simulation, by acting as a memorisable frame ID. Contiguous held-out '        'block is the only defence. Do not report a random-frame split.',        'Gate MAGNITUDE is not a diagnostic: a zero-information condition drove it to 0.306 '        'while an informative one reached 0.096. Use the condition-driven fraction.',        'Selecting the best of N epochs by the reported metric inflates it. Run E plateau was '        '0.112 +/- 0.037; E[min of 7] = 0.063. Epoch is selected here on a VALIDATION slice '        'carved out of the TRAIN block, never on the test block.',    ],    'registered_at_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),}if os.path.isfile(PREREG_PATH):    with open(PREREG_PATH) as f:        PREREG = json.load(f)    print(f'Loaded EXISTING pre-registration ({PREREG["registered_at_utc"]}) - not overwritten.')else:    with open(PREREG_PATH, 'w') as f:        json.dump(PREREG, f, indent=2)    print(f'Pre-registration WRITTEN to {PREREG_PATH}')print(json.dumps(PREREG['primary_endpoint'], indent=2))

## 2. Data — local copy and the official contiguous splitDrive's FUSE mount has high per-file-open latency and every frame is read once by thecaching pass, so the identity is extracted to local disk first (same pattern as theexisting notebooks).The split is read from `transforms_train.json` / `transforms_val.json` rather thanhardcoded, and asserted to be contiguous and non-overlapping. For `May` this gives5520 train / 553 test.

In [ ]:
os.makedirs(LOCAL_ROOT, exist_ok=True)def ensure_local_identity(name):    local = os.path.join(LOCAL_ROOT, name)    if os.path.isdir(os.path.join(local, 'gt_imgs')):        print(f'{name}: already local at {local}')        return local    tar = os.path.join(DATA_ROOT, f'{name}_train.tar')    src = os.path.join(DATA_ROOT, name)    if os.path.isfile(tar):        print(f'{name}: extracting {tar} -> {LOCAL_ROOT}')        with tarfile.open(tar) as t:            t.extractall(LOCAL_ROOT)    elif os.path.isdir(src):        print(f'{name}: copying {src} -> {local} (no tar found; this is slow)')        shutil.copytree(src, local)    else:        raise FileNotFoundError(f'neither {tar} nor {src} exists')    assert os.path.isdir(os.path.join(local, 'gt_imgs')), f'{local}/gt_imgs missing after extract'    return localLOCAL_PATH = ensure_local_identity(IDENTITY)def sorted_numeric_jpgs(dir_path):    '''Duplicate-upload tolerant, sorts by integer frame number. Plain    sorted(glob('*.jpg')) on unpadded names scrambles temporal order by ~1000    frames per window - that was a real bug in this project.'''    kept, skipped = [], 0    for p in glob.glob(os.path.join(dir_path, '*.jpg')):        stem = os.path.splitext(os.path.basename(p))[0]        try:            int(stem); kept.append(p)        except ValueError:            skipped += 1    kept.sort(key=lambda p: int(os.path.splitext(os.path.basename(p))[0]))    return kept, skippedFRAME_PATHS, n_skipped = sorted_numeric_jpgs(os.path.join(LOCAL_PATH, 'gt_imgs'))N_FRAMES = len(FRAME_PATHS)print(f'{N_FRAMES} frames ({n_skipped} non-numeric names skipped)')

In [ ]:
def official_split(drive_identity_dir, n_frames):    '''Read SyncTalk's own train/val split. Falls back to int(n*10/11), the    convention used byte-identically in AD-NeRF / RAD-NeRF / ER-NeRF / SyncTalk /    TalkingGaussian.'''    tr = os.path.join(drive_identity_dir, 'transforms_train.json')    va = os.path.join(drive_identity_dir, 'transforms_val.json')    if os.path.isfile(tr) and os.path.isfile(va):        with open(tr) as f: t = json.load(f)['frames']        with open(va) as f: v = json.load(f)['frames']        tr_ids = sorted(int(x['img_id']) for x in t)        va_ids = sorted(int(x['img_id']) for x in v)        src = 'transforms_*.json'    else:        cut = int(n_frames * 10 / 11)        tr_ids, va_ids = list(range(cut)), list(range(cut, n_frames))        src = 'int(n*10/11) fallback'    tr_ids = [i for i in tr_ids if i < n_frames]    va_ids = [i for i in va_ids if i < n_frames]    return np.array(tr_ids), np.array(va_ids), srcTRAIN_IDX, TEST_IDX, split_src = official_split(os.path.join(DATA_ROOT, IDENTITY), N_FRAMES)# hard asserts - this is the one thing that must not be wrongassert len(set(TRAIN_IDX) & set(TEST_IDX)) == 0, 'train/test overlap'assert TRAIN_IDX.max() < TEST_IDX.min(), 'split is not chronological'assert np.all(np.diff(TEST_IDX) == 1), 'test block is not contiguous'gap = int(TEST_IDX.min() - TRAIN_IDX.max())print(f'split source: {split_src}')print(f'train: {TRAIN_IDX.min()}..{TRAIN_IDX.max()}  ({len(TRAIN_IDX)} frames)')print(f'test : {TEST_IDX.min()}..{TEST_IDX.max()}  ({len(TEST_IDX)} frames)')print(f'temporal gap train->test: {gap} frame(s) = {gap/25*1000:.0f} ms')# a validation slice carved from the END of the train block, for epoch selection.# The test block is never touched until the verdict cell.VAL_LEN = 553VAL_IDX  = TRAIN_IDX[-VAL_LEN:]FIT_IDX  = TRAIN_IDX[:-VAL_LEN]print(f'fit  : {FIT_IDX.min()}..{FIT_IDX.max()} ({len(FIT_IDX)})   '      f'val: {VAL_IDX.min()}..{VAL_IDX.max()} ({len(VAL_IDX)})  [epoch selection]')

## 3. Stage 1 — cache the frozen FaRL representation`FarlFrozenFaceEncoder` copied verbatim from `talking_head_jepa.py` (2026-08-26version) so the cache is bit-comparable with Run D/E. Note what it returns: the**mean over patch tokens**, 768-d, at sequence position 0.Frames are preprocessed exactly as every other notebook in this project does it:`cv2.imread` (BGR) → RGB → `cv2.resize(img, (224,224))` (default `INTER_LINEAR`,whole-frame squash, no face crop) → `/255.0`, no CLIP normalization.One pass over 6073 frames, cached to Drive. Delete the `.npz` to force a rebuild.

In [ ]:
import urllib.requestclass FarlFrozenFaceEncoder(nn.Module):    '''Verbatim from talking_head_jepa.py (2026-08-26). Always frozen/eval.    Returns .last_hidden_state of shape (N, 1, 768) = the PATCH-TOKEN MEAN    (not 196 dense tokens, not the CLS token) at sequence position 0.'''    FARL_CHECKPOINT_URL = ('https://github.com/FacePerceiver/FaRL/releases/download/'                           'pretrained_weights/FaRL-Base-Patch16-LAIONFace20M-ep16.pth')    def __init__(self, embed_dim=768, checkpoint_dir=None):        super().__init__()        import clip        checkpoint_dir = checkpoint_dir or os.path.join(CODE_BASE, 'probing_cache', 'farl')        os.makedirs(checkpoint_dir, exist_ok=True)        ckpt_path = os.path.join(checkpoint_dir, 'FaRL-Base-Patch16-LAIONFace20M-ep16.pth')        if not os.path.isfile(ckpt_path):            print('downloading FaRL checkpoint (public GitHub release)...')            urllib.request.urlretrieve(self.FARL_CHECKPOINT_URL, ckpt_path)        assert os.path.getsize(ckpt_path) > 1e6, 'FaRL checkpoint download failed'        clip_model, _ = clip.load('ViT-B/16', device='cpu')        state = torch.load(ckpt_path, map_location='cpu')        missing, unexpected = clip_model.load_state_dict(state['state_dict'], strict=False)        print(f'FaRL -> CLIP ViT-B/16: {len(missing)} missing, {len(unexpected)} unexpected '              f'(expected: FaRL only overwrites the shared visual+text backbone)')        self.visual = clip_model.visual.float()        self.embed_dim = embed_dim        for p in self.visual.parameters():            p.requires_grad_(False)        self.visual.eval()    def train(self, mode=True):        return super().train(False)    def _dense(self, pixels):        '''Full token sequence (N, 1+196, 768) after ln_post on ALL tokens.'''        x = pixels.type(self.visual.conv1.weight.dtype)        x = self.visual.conv1(x)        x = x.reshape(x.shape[0], x.shape[1], -1).permute(0, 2, 1)        cls_tok = self.visual.class_embedding.to(x.dtype) + torch.zeros(            x.shape[0], 1, x.shape[-1], dtype=x.dtype, device=x.device)        x = torch.cat([cls_tok, x], dim=1)        x = x + self.visual.positional_embedding.to(x.dtype)        x = self.visual.ln_pre(x)        x = x.permute(1, 0, 2)        x = self.visual.transformer(x)        x = x.permute(1, 0, 2)        x = self.visual.ln_post(x)     # ALL tokens, not just x[:, 0]        return x    def forward(self, pixels, interpolate_pos_encoding=True):        with torch.no_grad():            x = self._dense(pixels)            patch_mean = x[:, 1:].mean(dim=1).float()      # (N, 768)  <-- THE latent        class Output: pass        out = Output()        out.last_hidden_state = patch_mean.unsqueeze(1)     # (N, 1, 768)        return out

In [ ]:
!pip install -q ftfy regex!pip install -q git+https://github.com/openai/CLIP.git

In [ ]:
def load_pixels(paths, idxs, size=224):    '''The project's canonical preprocessing. BGR->RGB, squash-resize to 224,    /255.0, NCHW. No crop, no CLIP mean/std.'''    imgs = []    for i in idxs:        img = cv2.imread(paths[int(i)])        assert img is not None, f'failed to read {paths[int(i)]}'        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)        img = cv2.resize(img, (size, size))        imgs.append(img)    arr = np.stack(imgs).astype(np.uint8)    px = torch.from_numpy(arr.astype(np.float32) / 255.0).permute(0, 3, 1, 2)    return px, arrFARL_CACHE = os.path.join(CACHE_DIR, f'{IDENTITY}_farl_patchmean_all.npz')if os.path.isfile(FARL_CACHE):    z = np.load(FARL_CACHE)    FARL = z['patch_mean']    assert len(FARL) == N_FRAMES, f'cache has {len(FARL)} rows, {N_FRAMES} frames on disk - delete and rebuild'    print(f'loaded cache {FARL.shape} from {FARL_CACHE}')else:    enc = FarlFrozenFaceEncoder().to(device).eval()    out = np.empty((N_FRAMES, 768), dtype=np.float32)    B = 32    for b0 in tqdm(range(0, N_FRAMES, B), desc='FaRL cache'):        idxs = np.arange(b0, min(b0 + B, N_FRAMES))        px, _ = load_pixels(FRAME_PATHS, idxs)        out[idxs] = enc(px.to(device)).last_hidden_state[:, 0].cpu().numpy()    FARL = out    np.savez_compressed(FARL_CACHE, patch_mean=FARL, n_frames=N_FRAMES)    print(f'wrote {FARL.shape} -> {FARL_CACHE}')    del enc; gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else Noneprint(f'FaRL latent: {FARL.shape[1]} floats per frame vs {224*224*3} image values '      f'=> {224*224*3/FARL.shape[1]:.0f}x compression')

## 4. Stage 2 — audio feature sanity, and is audio learnable here at allTwo free checks that decide whether anything downstream can work.`aud_ave.npy` is **not** an average. `ave` = Audio-Visual Encoder: each 512-d row isWav2Lip's LRS2-trained lip-sync tower encoding a 0.2 s mel window(`start_idx = int(80.*(frame/25.))`, `end_idx = start_idx + 16`). It is a goodfeature — SyncTalk's own ablation has it beating HuBERT on LMD (2.8032 vs 3.3961).The problem is never the encoder; it is the 0.2 s window versus the 0.5–0.85 sreceptive field that AD-NeRF / RAD-NeRF / ER-NeRF / SyncTalk all converge on.

In [ ]:
AUD = np.load(os.path.join(LOCAL_PATH, 'aud_ave.npy')).astype(np.float32)print(f'aud_ave: {AUD.shape}  frames on disk: {N_FRAMES}  (delta {len(AUD) - N_FRAMES})')# extract_ave_features repeats the first/last frame, so |n_aud - n_frames| <= 6 is expectedassert abs(len(AUD) - N_FRAMES) <= 6, 'audio/frame count mismatch beyond the known padding'N_USABLE = min(len(AUD), N_FRAMES)print(f'usable frames: {N_USABLE}')def check_audio_features(c, label=''):    print(f'--- audio feature sanity {label} ---')    mean, std = c.mean(0), c.std(0)    ratio = np.linalg.norm(std) / (np.linalg.norm(mean) + 1e-9)    print(f'||std||/||mean|| = {ratio:.4f}')    if ratio < 0.5:        print('  >>> large constant offset relative to temporal variation.')        print("      AdaLN's signal is W.(c - c_bar); a constant offset is structurally")        print('      indistinguishable from the modulation bias. Free fix: whiten.')    s = np.linalg.svd(c - mean, compute_uv=False)    eff_rank = int((s > 0.01 * s.max()).sum())    print(f'effective rank = {eff_rank} / {c.shape[1]}')    if eff_rank < 10:        print('  >>> condition is near-degenerate; nothing can be driven by it.')    cz = (c - mean) / (std + 1e-9)    ac1 = float((cz[:-1] * cz[1:]).mean())    print(f'lag-1 autocorrelation = {ac1:.4f}  (high => little new signal per frame)')    return {'std_over_mean': float(ratio), 'effective_rank': eff_rank, 'lag1_autocorr': ac1}audio_stats = check_audio_features(AUD[:N_USABLE], f'({IDENTITY}, all frames)')

In [ ]:
def persistence_share(emb, label=''):    '''Fraction of embedding variance NOT explained by "copy the previous frame".    If this is a few percent, an MSE objective is dominated by the copy solution and    no conditioning architecture will save it - fix the objective instead.    (Wav2Lip: the lip region is <4% of total reconstruction loss, which is exactly    why L1/MSE generators ignore audio. Same arithmetic in embedding space.)'''    e = torch.as_tensor(emb).float()    mse_persist = ((e[1:] - e[:-1]) ** 2).mean().item()    var_total = e.var(0).mean().item()    frac = mse_persist / (var_total + 1e-9)    print(f'{label:12s} persistence MSE {mse_persist:.6f} | var {var_total:.6f} | '          f'NOT explained by persistence {frac:.4f}')    if frac < 0.05:        print('             >>> under 5%: loss dominated by "copy the last frame". '              'Predict the RESIDUAL e_t - e_{t-1}.')    return fracps_train = persistence_share(FARL[TRAIN_IDX], 'train')ps_test  = persistence_share(FARL[TEST_IDX],  'test')

## 5. Stage 3 — Run F: retrain the predictor side on the train block onlySame architecture and loss as Run E (frozen FaRL, scheduled sampling, 3 chained hopsof 4 steps), with two differences:1. it trains on the **cached** 768-d features, so an epoch takes seconds;2. it trains on frames **0–4966** only (`FIT_IDX`), selecting the epoch on   `VAL_IDX` (4967–5519). The test block 5520–6072 is untouched.The loss is `compute_talking_head_loss_scheduled` from `JEPA_TalkingHead_EncoderRetrain.ipynb`,with the audio alignment that notebook actually used: `pred_act = act_emb[:, start:end]`— this hop's **future** audio. (Not `talking_head_jepa.py`'s `forward()`; see landmine 2.)

In [ ]:
class AudioFeatureEmbedder(nn.Module):    def __init__(self, audio_feature_dim, emb_dim, hidden_dim=512):        super().__init__()        self.net = nn.Sequential(nn.Linear(audio_feature_dim, hidden_dim), nn.GELU(),                                 nn.Linear(hidden_dim, emb_dim))    def forward(self, x):        return self.net(x)class SIGReg(nn.Module):    '''Variance/covariance anti-collapse term. Imported from the repo if present so    the objective matches Run E exactly; the fallback is only for a bare runtime.'''    def __init__(self, eps=1e-4):        super().__init__()        self.eps = eps    def forward(self, emb):                      # emb: (T, B, D) per the repo's call site        e = rearrange(emb, 't b d -> (t b) d')        e = e - e.mean(0, keepdim=True)        std = torch.sqrt(e.var(0) + self.eps)        var_term = F.relu(1.0 - std).mean()        n, d = e.shape        cov = (e.T @ e) / max(n - 1, 1)        off = cov - torch.diag(torch.diag(cov))        cov_term = off.pow(2).sum() / d        return var_term + cov_termtry:    from talking_head_jepa import SIGReg as RepoSIGReg     # prefer the repo's own    SIGReg = RepoSIGReg    print('using repo SIGReg')except Exception as e:    print('repo SIGReg not importable, using local equivalent:', e)class CachedJEPA(nn.Module):    '''Run D/E's trainable half, operating on cached frozen-encoder features.    projector / audio_embedder / predictor / pred_proj are exactly the Run D/E    modules with the same dims, so a Run E checkpoint loads straight into it.'''    def __init__(self, feat_dim=768, audio_dim=AUDIO_DIM, embed_dim=EMBED_DIM,                 hidden_dim=HIDDEN_DIM, history_size=HISTORY_SIZE):        super().__init__()        self.history_size = history_size        self.embed_dim = embed_dim        self.projector = MLP(input_dim=feat_dim, output_dim=embed_dim,                             hidden_dim=1024, norm_fn=nn.BatchNorm1d)        self.audio_embedder = AudioFeatureEmbedder(audio_feature_dim=audio_dim, emb_dim=embed_dim)        self.predictor = ARPredictor(num_frames=history_size, input_dim=embed_dim,                                     hidden_dim=hidden_dim, output_dim=embed_dim,                                     depth=6, heads=8, mlp_dim=hidden_dim * 4)        self.pred_proj = MLP(input_dim=embed_dim, output_dim=embed_dim,                             hidden_dim=1024, norm_fn=nn.BatchNorm1d)    def project(self, feats):                 # (B, T, 768) -> (B, T, D)        b = feats.size(0)        e = self.projector(rearrange(feats, 'b t d -> (b t) d'))        return rearrange(e, '(b t) d -> b t d', b=b)    def embed_audio(self, aud):               # (B, T, A) -> (B, T, D)        return self.audio_embedder(aud)    def predict(self, emb, act_emb):          # (B, Tctx, D), (B, Tpred, D) -> (B, Tpred, D)        p = self.predictor(emb, act_emb)        b = emb.size(0)        p = self.pred_proj(rearrange(p, 'b t d -> (b t) d'))        return rearrange(p, '(b t) d -> b t d', b=b)model = CachedJEPA().to(device)n_tr = sum(p.numel() for p in model.parameters() if p.requires_grad)print(f'trainable params: {n_tr:,}  (the 86M frozen FaRL params live in the cache, not here)')

In [ ]:
# --- AdaLN-zero check: at init the predictor is a pure persistence map ---------# ConditionalBlock zero-inits adaLN_modulation[-1] weight AND bias, so at step 0# ARPredictor computes LayerNorm(ctx + pos_emb): audio enters only through gates# that start at exactly zero. Worth printing once so a "predictor does nothing"# result at epoch 0 is read as expected, not as a bug.with torch.no_grad():    blk = model.predictor.transformer.layers[0]    last = blk.adaLN_modulation[-1]    print(f'block0 adaLN last layer: |W|={last.weight.abs().max().item():.3e} '          f'|b|={last.bias.abs().max().item():.3e}   (both ~0 at init = AdaLN-zero)')print(f'embed_dim == hidden_dim == {EMBED_DIM} -> the predictor input/cond/output '      f'projections are nn.Identity()')print()print('Consequence worth knowing: because that weight is exactly 0, the audio_embedder')print('receives EXACTLY ZERO gradient on optimizer step 0 - the audio path is literally')print('disconnected at init. It reconnects on step 1, once adaLN_modulation has moved off')print('zero (its own gradient is grad_out . cond^T, which is not zero). Verified offline.')print('So "audio does nothing" at epoch 0 is expected; only later epochs are evidence.')

In [ ]:
# --- windows over cached features --------------------------------------------def make_windows(frame_idx, seq_len):    '''All start positions s such that [s, s+seq_len) lies entirely inside the    given contiguous frame block. No window ever straddles the split.'''    lo, hi = int(frame_idx.min()), int(frame_idx.max())    hi = min(hi, N_USABLE - 1)    return np.arange(lo, hi - seq_len + 2)FIT_W  = make_windows(FIT_IDX,  SEQ_LEN)VAL_W  = make_windows(VAL_IDX,  SEQ_LEN)TEST_W = make_windows(TEST_IDX, SEQ_LEN)print(f'windows (seq_len={SEQ_LEN}): fit {len(FIT_W)}  val {len(VAL_W)}  test {len(TEST_W)}')assert FIT_W.max() + SEQ_LEN <= VAL_W.min() + 1, 'fit windows leak into val'assert VAL_W.max() + SEQ_LEN <= TEST_W.min() + 1, 'val windows leak into test'FARL_T = torch.from_numpy(FARL[:N_USABLE]).float()AUD_T  = torch.from_numpy(AUD[:N_USABLE]).float()def gather_batch(starts, seq_len, audio=None):    '''-> feats (B, T, 768), aud (B, T, A) on device.'''    idx = torch.from_numpy(np.stack([np.arange(s, s + seq_len) for s in starts]))    f = FARL_T[idx]    a = (AUD_T if audio is None else audio)[idx]    return f.to(device), a.to(device)

In [ ]:
def scheduled_sampling_prob(step, total_steps, start_prob=0.0, end_prob=0.5, warmup_frac=0.3):    '''Run E's ramp: 0 -> 0.5 self-feeding over the first 30% of training, then flat.'''    if total_steps <= 0:        return start_prob    warm = max(1, int(warmup_frac * total_steps))    if step >= warm:        return end_prob    return start_prob + (end_prob - start_prob) * (step / warm)def loss_scheduled(model, feats, aud, step, total_steps, num_preds=NUM_PREDS,                   n_hops=N_HOPS_TRAIN, sigreg=None, sigreg_weight=1.0, train=True):    '''compute_talking_head_loss_scheduled, on cached features.    Audio alignment is the EncoderRetrain notebook's: pred_act = act_emb[:, start:end]    -> this hop's FUTURE audio.'''    emb = model.project(feats)                     # (B, T, D)    act = model.embed_audio(aud)                   # (B, T, D)    ctx_len = model.history_size    p_sample = scheduled_sampling_prob(step, total_steps) if train else 0.0    ctx = emb[:, :ctx_len]    preds, tgts, hop_losses = [], [], []    for hop in range(n_hops):        start = ctx_len + hop * num_preds        end = start + num_preds        pred_act = act[:, start:end]               # future audio for this hop        tgt = emb[:, start:end].detach()        pred = model.predict(ctx, pred_act)        hop_losses.append((pred - tgt).pow(2).mean())        preds.append(pred); tgts.append(tgt)        real_next = emb[:, start:end].detach()        if train and p_sample > 0:            # per-SAMPLE mix, matching Run E            use_pred = (torch.rand(pred.size(0), 1, 1, device=pred.device) < p_sample).float()            ctx = use_pred * pred.detach() + (1 - use_pred) * real_next        else:            ctx = real_next    pred_loss = torch.stack(hop_losses).mean()    out = {'pred_loss': pred_loss, 'hop_losses': [h.item() for h in hop_losses],           'p_sample': p_sample,           'pred_emb': torch.cat(preds, 1), 'tgt_emb': torch.cat(tgts, 1)}    if sigreg is not None:        sr = sigreg(emb.transpose(0, 1))        out['sigreg_loss'] = sr        out['loss'] = pred_loss + sigreg_weight * sr    else:        out['loss'] = pred_loss    return out

In [ ]:
CKPT_PATH = os.path.join(CKPT_DIR, f'{RUN_TAG}_best.pth')RETRAIN = True   # set False to load the existing Run F checkpoint insteadsigreg = SIGReg().to(device)opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)rng = np.random.default_rng(SEED)steps_per_epoch = max(1, len(FIT_W) // BATCH_SIZE)TOTAL_STEPS = N_EPOCHS * steps_per_epoch@torch.no_grad()def eval_windows(model, W, batch=64):    '''Mean 12-step (3-hop, real context) MSE - the epoch-selection metric.'''    model.eval(); tot, n = 0.0, 0    for b0 in range(0, len(W), batch):        starts = W[b0:b0 + batch]        f, a = gather_batch(starts, SEQ_LEN)        o = loss_scheduled(model, f, a, 0, 1, train=False)        tot += o['pred_loss'].item() * len(starts); n += len(starts)    model.train()    return tot / max(n, 1)if RETRAIN:    history, best = [], (float('inf'), -1)    gstep = 0    model.train()    for ep in range(N_EPOCHS):        perm = rng.permutation(FIT_W)        run = 0.0        for b0 in tqdm(range(0, steps_per_epoch * BATCH_SIZE, BATCH_SIZE),                       desc=f'epoch {ep+1}/{N_EPOCHS}', leave=False):            starts = perm[b0:b0 + BATCH_SIZE]            if len(starts) < BATCH_SIZE:                break            f, a = gather_batch(starts, SEQ_LEN)            o = loss_scheduled(model, f, a, gstep, TOTAL_STEPS, sigreg=sigreg)            opt.zero_grad(); o['loss'].backward(); opt.step()            run += o['pred_loss'].item(); gstep += 1        val = eval_windows(model, VAL_W)        history.append({'epoch': ep + 1, 'train_pred_loss': run / steps_per_epoch,                        'val_pred_loss': val, 'p_sample': o['p_sample']})        star = ''        if val < best[0]:            best = (val, ep + 1)            torch.save({'epoch': ep + 1, 'model_state_dict': model.state_dict(),                        'val_pred_loss': val, 'run_tag': RUN_TAG}, CKPT_PATH)            star = '  <- best'        print(f'epoch {ep+1:2d}  train {run/steps_per_epoch:.5f}  val {val:.5f}  '              f'p_sample {o["p_sample"]:.2f}{star}')    print(f'\nbest epoch {best[1]} val {best[0]:.5f} -> {CKPT_PATH}')    print('NOTE: epoch selected on VAL (inside the train block). The test block is still untouched.')else:    history = []ck = torch.load(CKPT_PATH, map_location=device)model.load_state_dict(ck['model_state_dict'])model.eval()print(f'loaded {RUN_TAG} epoch {ck["epoch"]} (val {ck["val_pred_loss"]:.5f})')

In [ ]:
if history:    h = history    plt.figure(figsize=(6, 3.2))    plt.plot([x['epoch'] for x in h], [x['train_pred_loss'] for x in h], label='train')    plt.plot([x['epoch'] for x in h], [x['val_pred_loss'] for x in h], label='val (in train block)')    plt.axvline(ck['epoch'], ls='--', c='k', lw=1, label=f'selected ep{ck["epoch"]}')    plt.xlabel('epoch'); plt.ylabel('12-step pred MSE'); plt.legend(); plt.grid(alpha=.3)    plt.title(f'{RUN_TAG}'); plt.tight_layout(); plt.show()    vals = np.array([x['val_pred_loss'] for x in h])    tail = vals[max(0, len(vals) - 7):]    print(f'plateau (last {len(tail)} epochs): mean {tail.mean():.4f} +/- {tail.std(ddof=1):.4f}')    print(f'min of those {len(tail)}: {tail.min():.4f}')    print('The gap between the plateau mean and the min is SELECTION OPTIMISM. Report the '          'plateau mean +/- SD as the honest estimate of this configuration; the min is what '          'the selected checkpoint happens to score.')

## 6. Stage 4 — condition-driven fractionReplaces "log the gate magnitude", which is not a diagnostic: in simulation acondition carrying **literally zero information** drove gate magnitude to 0.306 whilean informative one reached only 0.096. Magnitude measures whether the transformerturned itself on via the modulation *bias*.What matters is how much of the modulation **varies across different audio inputs**versus how much is a learned constant:$$\mathrm{CDF} = \frac{\mathrm{std}_{\text{across conditions}}}{\mathrm{std}_{\text{across conditions}} + |\text{mean}|}$$| CDF | reading ||---|---|| < 0.02 | dead — modulation is a learned constant (zero-information control sat at exactly 0.000) || 0.02–0.15 | weak, bias-dominated — believe the shuffle test, not this || 0.3–0.8 | alive (the informative control held 0.66–0.74) |

In [ ]:
@torch.no_grad()def condition_driven_fraction(model, cond_batch, verbose=True):    names = ('shift1', 'scale1', 'gate1', 'shift2', 'scale2', 'gate2')    rows = []    for bi, block in enumerate(model.predictor.transformer.layers):        out = block.adaLN_modulation(cond_batch)        if verbose: print(f'-- block {bi} --')        for name, t in zip(names, out.chunk(6, dim=1)):            cond_std = t.std(0).mean().item()            const_mag = t.mean(0).abs().mean().item()            cdf = cond_std / (cond_std + const_mag + 1e-9)            flag = ''            if 'gate' in name:                flag = ('   <<< DEAD' if cdf < 0.02 else '   <<< weak' if cdf < 0.15                        else '   <<< alive' if cdf > 0.30 else '')            rows.append({'block': bi, 'row': name, 'abs_mod': t.abs().mean().item(),                         'cond_std': cond_std, 'const_mag': const_mag, 'cdf': cdf})            if verbose:                print(f'  {name:7s} |mod|={t.abs().mean():.4f}  cond_std={cond_std:.4f}  '                      f'CDF={cdf:.3f}{flag}')    return rows# N DIFFERENT held-out audio windows, embedded exactly as the predictor sees them_pool = TEST_IDX[:len(TEST_IDX) - 1]sel = np.random.default_rng(SEED).choice(_pool, size=min(256, len(_pool)), replace=False)cond = model.embed_audio(AUD_T[sel].to(device))cdf_rows = condition_driven_fraction(model, cond)gate_cdfs = [r['cdf'] for r in cdf_rows if 'gate' in r['row']]print(f'\ngate CDF: min {min(gate_cdfs):.3f}  median {np.median(gate_cdfs):.3f}  max {max(gate_cdfs):.3f}')print('|mod| column is printed only to show it is uninformative. Do not threshold on it.')

## 7. Stage 5 — the held-out verdict: real vs shuffled vs zeroed audioThe primary endpoint. Evaluated on the contiguous test block only, with real contextembeddings, at two horizons:- **hop 1** — 4 steps ahead (160 ms), the horizon the model was trained on per hop;- **hop 3** — 12 steps ahead (480 ms) with the model's own predictions fed back, the  horizon scheduled sampling was added for.Four nulls. `shuffled` permutes audio rows across the whole test block, which destroysthe frame↔audio correspondence while preserving the marginal distribution exactly — soa model that has merely learned "audio is roughly this shape" is not rewarded.`persistence` and `train_mean` bound how much of any improvement is trivially available.`audio_share` is the number worth quoting: of the model's improvement over persistence,what fraction disappears when the audio is wrong.

In [ ]:
@torch.no_grad()def rollout_eval(model, W, audio_variant='real', n_hops=N_HOPS_TRAIN, batch=64, seed=SEED):    '''Per-window per-hop MSE. audio_variant in {real, shuffled, zeroed}.    shuffled permutes audio rows across the WHOLE test block, once, with a fixed seed.'''    model.eval()    if audio_variant == 'real':        AUD_USE = AUD_T    elif audio_variant == 'zeroed':        AUD_USE = torch.zeros_like(AUD_T)    elif audio_variant == 'shuffled':        lo, hi = int(W.min()), int(W.max()) + SEQ_LEN        perm = np.random.default_rng(seed).permutation(np.arange(lo, hi))        AUD_USE = AUD_T.clone()        AUD_USE[lo:hi] = AUD_T[torch.from_numpy(perm)]    else:        raise ValueError(audio_variant)    per_hop = [[] for _ in range(n_hops)]    per_hop_persist = [[] for _ in range(n_hops)]    per_hop_mean = [[] for _ in range(n_hops)]    train_mean_emb = None    for b0 in range(0, len(W), batch):        starts = W[b0:b0 + batch]        f, a = gather_batch(starts, SEQ_LEN, audio=AUD_USE)        emb = model.project(f)        act = model.embed_audio(a)        if train_mean_emb is None:            fm, _ = gather_batch(FIT_W[:512], SEQ_LEN)            train_mean_emb = model.project(fm).reshape(-1, EMBED_DIM).mean(0, keepdim=True)        ctx_len = model.history_size        ctx = emb[:, :ctx_len]        last_real = emb[:, ctx_len - 1:ctx_len]                 # (B, 1, D)        for hop in range(n_hops):            s, e = ctx_len + hop * NUM_PREDS, ctx_len + (hop + 1) * NUM_PREDS            tgt = emb[:, s:e]            pred = model.predict(ctx, act[:, s:e])            per_hop[hop].append((pred - tgt).pow(2).mean(dim=(1, 2)).cpu().numpy())            per_hop_persist[hop].append((last_real.expand_as(tgt) - tgt).pow(2)                                        .mean(dim=(1, 2)).cpu().numpy())            per_hop_mean[hop].append((train_mean_emb.unsqueeze(0).expand_as(tgt) - tgt)                                     .pow(2).mean(dim=(1, 2)).cpu().numpy())            ctx = emb[:, s:e] if hop == n_hops - 1 else pred.detach()   # self-feed for hop 2+    return {'model': [np.concatenate(x) for x in per_hop],            'persistence': [np.concatenate(x) for x in per_hop_persist],            'train_mean': [np.concatenate(x) for x in per_hop_mean]}def boot_ci(a, b, n_boot=10000, seed=0):    '''Bootstrap 95% CI for 100*(mean(b)-mean(a))/mean(a), paired over windows.'''    rng = np.random.default_rng(seed)    n = len(a)    idx = rng.integers(0, n, size=(n_boot, n))    d = 100.0 * (b[idx].mean(1) - a[idx].mean(1)) / (a[idx].mean(1) + 1e-12)    return float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))R = {v: rollout_eval(model, TEST_W, audio_variant=v) for v in ('real', 'shuffled', 'zeroed')}R_IN = {v: rollout_eval(model, FIT_W[:len(TEST_W)], audio_variant=v) for v in ('real', 'shuffled')}print(f'evaluated {len(TEST_W)} held-out windows and {len(TEST_W)} in-sample windows')

In [ ]:
def report(R, hop, label='held-out'):    real = R['real']['model'][hop]    shuf = R['shuffled']['model'][hop]    zero = R['zeroed']['model'][hop] if 'zeroed' in R else None    pers = R['real']['persistence'][hop]    tmean= R['real']['train_mean'][hop]    d_shuf = 100.0 * (shuf.mean() - real.mean()) / real.mean()    lo, hi = boot_ci(real, shuf, seed=hop)    beats = pers.mean() - real.mean()    share = (shuf.mean() - real.mean()) / beats if beats > 0 else float('nan')    print(f'--- hop {hop+1} ({(hop+1)*NUM_PREDS} steps = {(hop+1)*NUM_PREDS/25*1000:.0f} ms), {label} '          f'[n={len(real)} windows] ---')    print(f'  MSE real audio       {real.mean():.5f}')    print(f'  MSE shuffled audio   {shuf.mean():.5f}   delta {d_shuf:+.1f}%  '          f'[95% CI {lo:+.1f}, {hi:+.1f}]')    if zero is not None:        print(f'  MSE zeroed audio     {zero.mean():.5f}   '              f'delta {100.0*(zero.mean()-real.mean())/real.mean():+.1f}%')    if beats > 0:        print(f'  MSE persistence null {pers.mean():.5f}   '              f'model is {100.0*beats/pers.mean():.1f}% BETTER than copying the last frame')    else:        print(f'  MSE persistence null {pers.mean():.5f}   '              f'>>> model is {100.0*(-beats)/pers.mean():.1f}% WORSE than copying the last '              f'frame - nothing downstream is meaningful until this is fixed')    print(f'  MSE train-mean null  {tmean.mean():.5f}')    if beats > 0:        print(f'  audio_share = (shuf-real)/(persist-real) = {share:.3f}   '              f'(fraction of the gain over persistence that is audio-attributable)')    else:        print('  audio_share = n/a (undefined when the model loses to persistence)')    return {'hop': hop + 1, 'n_windows': int(len(real)),            'mse_real': float(real.mean()), 'mse_shuffled': float(shuf.mean()),            'mse_zeroed': float(zero.mean()) if zero is not None else None,            'mse_persistence': float(pers.mean()), 'mse_train_mean': float(tmean.mean()),            'delta_shuffled_pct': float(d_shuf), 'delta_ci95': [lo, hi],            'audio_share': float(share)}held = [report(R, h, 'HELD-OUT (5520-6072)') for h in range(N_HOPS_TRAIN)]print()insample = [report(R_IN, h, 'in-sample (train block)') for h in range(N_HOPS_TRAIN)]

In [ ]:
h1, i1 = held[0], insample[0]print('=== leakage check: the same metric, in-sample vs held-out ===')print(f'  hop-1 delta   in-sample {i1["delta_shuffled_pct"]:+.1f}%   '      f'held-out {h1["delta_shuffled_pct"]:+.1f}%')print(f'  hop-1 MSE     in-sample {i1["mse_real"]:.5f}      held-out {h1["mse_real"]:.5f}   '      f'({100*(h1["mse_real"]-i1["mse_real"])/i1["mse_real"]:+.0f}%)')print()print('A large in-sample delta with a small held-out delta is the signature of the trap:')print('the condition acting as a memorisable per-frame ID rather than as audio. On 6000')print('frames of one speaker that is entirely feasible, and it is exactly what Run D/E')print('could not rule out - they had no held-out frames at all.')

## 8. Verdict against the pre-registration

In [ ]:
th = PREREG['thresholds']primary = held[0]['delta_shuffled_pct']ci_lo = held[0]['delta_ci95'][0]beats_persistence = held[0]['mse_persistence'] > held[0]['mse_real']if not beats_persistence:    verdict, action = ('FAIL',        'The model loses to the persistence null on held-out frames: copying the last real '        'context embedding predicts better than the predictor does. Any shuffled-audio delta '        'is measured against a broken baseline and means nothing. Fix this first - it usually '        'means the objective is dominated by the copy solution (see persistence_share above) '        'or that the projector output has drifted between train and test.')elif primary >= th['delta_pass'] and ci_lo > th['delta_weak']:    verdict, action = 'PASS', 'Audio is load-bearing on held-out frames. S1 (decoder oracle) may proceed.'elif primary >= th['delta_weak']:    verdict, action = ('WEAK',        'Some audio dependence, but not enough to build on. Apply the Tier-0 fixes in order '        '(1: whiten audio - free; 2: window audio +/-8 frames = 0.64 s, matching SyncTalk_2D; '        '3: predict the residual e_t - e_{t-1}) and re-run THIS notebook before any decoder work.')else:    verdict, action = ('FAIL',        'The predictor is not using audio. Do not build a decoder - it would faithfully decode '        'an audio-independent latent and the whole result would be a persistence model with a '        'renderer. Apply the Tier-0 fixes and re-run.')RESULT = {    'study_id': STUDY_ID, 'run_tag': RUN_TAG,    'evaluated_at_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),    'prereg_registered_at_utc': PREREG['registered_at_utc'],    'selected_epoch': int(ck['epoch']), 'val_pred_loss': float(ck['val_pred_loss']),    'split': {'train': [int(TRAIN_IDX.min()), int(TRAIN_IDX.max())],              'fit': [int(FIT_IDX.min()), int(FIT_IDX.max())],              'val': [int(VAL_IDX.min()), int(VAL_IDX.max())],              'test': [int(TEST_IDX.min()), int(TEST_IDX.max())]},    'audio_stats': audio_stats,    'persistence_share': {'train': float(ps_train), 'test': float(ps_test)},    'gate_cdf': {'min': float(min(gate_cdfs)), 'median': float(np.median(gate_cdfs)),                 'max': float(max(gate_cdfs))},    'cdf_rows': cdf_rows,    'heldout': held, 'in_sample': insample,    'training_history': history,    'primary_endpoint_value': float(primary),    'verdict': verdict,}path = os.path.join(RESULTS_DIR, f'{STUDY_ID}_{RUN_TAG}.json')with open(path, 'w') as f:    json.dump(RESULT, f, indent=2)print('=' * 78)print(f'PRIMARY ENDPOINT  shuffled-audio delta, hop 1, held-out : {primary:+.1f}%  '      f'[CI {ci_lo:+.1f}, {held[0]["delta_ci95"][1]:+.1f}]')print(f'pre-registered    pass >= {th["delta_pass"]}% with CI low > {th["delta_weak"]}%')print(f'VERDICT           {verdict}')print('=' * 78)print(action)print(f'\nwritten: {path}')

## 9. How to read each outcome**FAIL.** This is the most likely outcome and the most useful one. It means Run D/E'sreported rollout improvements were persistence plus in-sample memorisation, and itmeans the decoder question is premature. The three Tier-0 fixes, in cost order:1. **Whiten the audio** — free. `c = (c - c.mean(0)) / (c.std(0) + 1e-6)`, statistics   fit on the train block only. AdaLN's signal is `W·(c - c̄)`; a large constant offset   is structurally indistinguishable from the modulation bias, so the network can reach   the same function with a dead condition path.2. **Window the audio** — `features[i-8:i+8]` = 16 × 512 = 0.64 s, matching   `SyncTalk_2D/datasetsss.py` exactly. Run D/E fed one 0.2 s vector; the consensus   receptive field across AD-NeRF / RAD-NeRF / ER-NeRF / SyncTalk is 0.5–0.85 s per   predicted frame. This is a 16× increase in temporal context and needs   `audio_feature_dim = 8192`.3. **Predict the residual** `e_t - e_{t-1}` — deletes the persistence shortcut from the   objective and rescales the loss so audio-driven variance is a large fraction of the   target rather than a few percent. Two lines here, one change at inference   (cumulative sum from the last real context embedding).Re-run this notebook after each fix. It takes minutes now, so run them one at a timeand keep the four numbers per fix — that table *is* the ablation for a paper.**WEAK.** Same fixes, but report the honest magnitude. A weak-but-real effect is stillpublishable as a negative or a limitation; a weak effect dressed up as a strong one isthe failure mode this whole pre-registration exists to prevent.**PASS.** Move to `S1_Decoder_Oracle.ipynb`. Note that PASS here is a statement aboutthe *predictor*, not about visual quality — S1 asks the separate question of whetherthe 768-d latent can be inverted to RGB at all.**In every case**, copy `study_results/S0_audio_instrument_runF_trainblock_only.json`into the knowledge base next to the pre-registration. The pair (pre-registration,result) is the artifact; either alone is worth much less.